In [1]:
from llama_index.llms.anthropic import Anthropic
from llama_index.core.agent.workflow import (
    FunctionAgent,
    AgentWorkflow,
    AgentOutput,
    ToolCall,
    ToolCallResult,
)
from tavily import AsyncTavilyClient
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
llm = Anthropic(model="claude-opus-4-5-20251101", max_tokens=4096)

research_state = {
    "research_notes": {},
    "report_content": "",
    "review": "",
}

In [4]:
async def search_web(query: str) -> str:
    """Search the web for information on a topic."""
    client = AsyncTavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    result = await client.search(query)
    return str(result)


async def record_notes(notes: str, notes_title: str) -> str:
    """Record research notes with a title."""
    research_state["research_notes"][notes_title] = notes
    return f"Notes recorded under title: {notes_title}"


async def write_report(report_content: str) -> str:
    """Write the final report content."""
    research_state["report_content"] = report_content
    return "Report written successfully."


async def review_report(review: str) -> str:
    """Review the report and provide feedback."""
    research_state["review"] = review
    return f"Review recorded: {review}"

In [5]:
research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Searches the web for information and records notes.",
    system_prompt=(
        "You are the ResearchAgent. Search the web for information on the given topic "
        "and record your findings as notes using the record_notes tool. "
        "Once you have gathered enough information (at least 2-3 searches), "
        "hand off to the WriteAgent to write the report."
    ),
    llm=llm,
    tools=[search_web, record_notes],
    can_handoff_to=["WriteAgent"],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Writes reports based on research notes.",
    system_prompt=(
        "You are the WriteAgent. Write a concise report in markdown format based on "
        "what the ResearchAgent found. Use the write_report tool to save the report. "
        "Once written, hand off to the ReviewAgent for feedback."
    ),
    llm=llm,
    tools=[write_report],
    can_handoff_to=["ReviewAgent", "ResearchAgent"],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Reviews reports and provides feedback.",
    system_prompt=(
        "You are the ReviewAgent. Review the report and use the review_report tool "
        "to record your feedback. Approve the report if it's good, or request specific "
        "changes. If major changes are needed, hand off to the WriteAgent."
    ),
    llm=llm,
    tools=[review_report],
    can_handoff_to=["WriteAgent"],
)

In [6]:
agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
)

In [7]:
handler = agent_workflow.run(
    user_msg=(
        "Write a brief equity research report on HDFCBANK's stock performance "
        "in May 2026. Focus on top level executive changes happening and its impact and analyst reactions."
    )
)

current_agent = None
async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"Agent: {current_agent}")
        print(f"{'='*50}\n")
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print(f"Output: {event.response.content[:200]}...")
        if event.tool_calls:
            print(f"Tools: {[call.tool_name for call in event.tool_calls]}")
    elif isinstance(event, ToolCallResult):
        output = str(event.tool_output)[:150]
        print(f"Tool Result ({event.tool_name}): {output}...")


Agent: ResearchAgent

Output: I'll research HDFCBANK's stock performance in May 2026, focusing on executive changes, their impact, and analyst reactions. Let me search for this information....
Tools: ['search_web', 'search_web', 'search_web']
Tool Result (search_web): {'query': 'HDFCBANK stock performance May 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'http://www.hdfcsec.co...
Tool Result (search_web): {'query': 'HDFCBANK analyst ratings recommendations May 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https:/...
Tool Result (search_web): {'query': 'HDFCBANK executive changes leadership 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.sc...
Output: I found some useful information. Let me search for more specific details on the chairman resignation controversy and analyst reactions to these executive changes....
Tools: ['search_web', 'search_web']
Tool R

In [8]:
print("\n" + "="*50)
print("FINAL EQUITY RESEARCH REPORT")
print("="*50)
print(research_state["report_content"])


FINAL EQUITY RESEARCH REPORT
# HDFC Bank (NSE: HDFCBANK) — Equity Research Report
## May 2026 | Focus: Executive Changes & Market Impact

---

### Executive Summary

HDFC Bank, India's largest private sector lender, faced significant headwinds in May 2026 as the stock continued to trade under pressure following a major governance crisis. The stock traded in the **₹747–790 range** during May, approximately **25% below its 52-week high** of ₹1,020.50. The sell-off, which began in March 2026, was triggered by the sudden resignation of Part-Time Chairman Atanu Chakraborty and ongoing uncertainty around CEO succession.

---

### Stock Performance Snapshot — May 2026

| Metric | Value |
|--------|-------|
| Price Range (May 2026) | ₹747 – ₹790 |
| Market Capitalization | ~₹11.54 lakh crore |
| 52-Week High | ₹1,020.50 |
| YTD Performance (2026) | Down ~20-25% |
| Near 52-Week Low | ₹726.65 |

The stock underperformed the Nifty Private Bank Index (down ~3% YTD) by a significant margin, refle

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.anthropic import Anthropic
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow
from dotenv import load_dotenv

load_dotenv()